# Surface following

Surface following is a feature on Hamilton liquid handling robots that makes the pipette tip follow the surface of a liquid when aspirating (going down) or dispensing (going up).

When using surface following, the robot will automatically move the Z position of the pipette tip the user-specified distance. The amount of surface following required can be computed by comparing the liquid level before and after each aspiration or dispense. PyLabRobot can do this automatically when the height<>volume functions for the given containers are defined. You can also specify the liquid surface following distance manually.

It is useful to start the surface following only at the liquid level, so it is recommended to use [liquid level detection](./lld) with the surface following feature. (See below for syntax, which differs from the LLD tutorial). VENUS also supports surface following while doing LLD.

In PLR, when we have LLD + automatic surface following, we can go beyond VENUS by computing the surface following amount based on the precise location of liquid inside the container. This is necessary because the surface following amount is not _just_ a function of the volume of liquid aspirated or dispensed, _but also_ of the location of liquid inside the container (see below). By doing liquid level detection first to get the precise liquid level, we can then use that liquid level height to compute the surface following amount based on the requested volume _and_ location of liquid inside the container.

![](./img/surface_following/surface_following_distance.svg)

## Setup

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star import STAR
from pylabrobot.resources.hamilton import STARDeck

deck = STARDeck()
star = STAR(deck=deck)
await star.setup()

from pylabrobot.resources import (
    TIP_CAR_480_A00,
    PLT_CAR_L5AC_A00,
    CellTreat_96_wellplate_350ul_Ub,
    hamilton_96_tiprack_1000uL_filter,
)

tip_car = TIP_CAR_480_A00(name="tip carrier")
tip_car[0] = tr0 = hamilton_96_tiprack_1000uL_filter(name="tips_01")
deck.assign_child_resource(tip_car, rails=2)

plt_car = PLT_CAR_L5AC_A00(name="plate carrier")
plt_car[0] = plate = CellTreat_96_wellplate_350ul_Ub(name="plate")
deck.assign_child_resource(plt_car, rails=14)

## Automatic surface following

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star.pip_backend import STARPIPBackend

wells = plate["A1:H1"]
vols = [50] * len(wells)

You can probe the liquid height first using liquid level detection (capacitive), and then use automatic surface following for subsequent aspirations and dispenses as follows:

In [ ]:
async with star.pip.use_tips(tr0["A1:H1"], trash=star.deck.get_resource("trash"), discard=False):
    await star.pip.aspirate(
        wells,
        vols,
        backend_params=STARPIPBackend.AspirateParams(
            # Probe the liquid height before aspirating.
            probe_liquid_height=True,
            # Automatically adjust the following distance based on the probed liquid height.
            auto_surface_following_distance=True,
        ),
    )

    await star.pip.dispense(
        wells,
        vols,
        backend_params=STARPIPBackend.DispenseParams(
            probe_liquid_height=True,
            auto_surface_following_distance=True,
        ),
    )

You can also pass the liquid height directly to the aspiration and dispense methods, and still use automatic surface following. This can be useful when you cannot use LLD. Note that `liquid_height` is a frontend parameter on `star.pip.aspirate()`, while `auto_surface_following_distance` is a backend parameter.

In [ ]:
async with star.pip.use_tips(tr0["A1:H1"], trash=star.deck.get_resource("trash"), discard=False):
    await star.pip.aspirate(
        wells,
        vols,
        liquid_height=[10] * len(wells),  # in mm above the bottom of the well
        backend_params=STARPIPBackend.AspirateParams(
            auto_surface_following_distance=True,
        ),
    )

    await star.pip.dispense(
        wells,
        vols,
        liquid_height=[10] * len(wells),  # in mm above the bottom of the well
        backend_params=STARPIPBackend.DispenseParams(
            auto_surface_following_distance=True,
        ),
    )

## Manual surface following

To manually specify the surface following amount, you can use the `surface_following_distance` backend parameter. For example, to aspirate 50 uL with a surface following distance of 2 mm starting at the detected liquid level:

In [ ]:
async with star.pip.use_tips(tr0["A1:H1"], trash=star.deck.get_resource("trash"), discard=False):
    await star.pip.aspirate(
        wells,
        vols,
        backend_params=STARPIPBackend.AspirateParams(
            probe_liquid_height=True,
            surface_following_distance=[2] * len(wells),  # mm down after finding liquid
        ),
    )

    await star.pip.dispense(
        wells,
        vols,
        backend_params=STARPIPBackend.DispenseParams(
            probe_liquid_height=True,
            surface_following_distance=[2] * len(wells),  # mm up after finding liquid
        ),
    )